# Configuración

Utilizar este notebook para realizar consultas a BiblioIA.

### Importación de librerías y dependencias

In [ ]:
import os
import requests
import pandas as pd
import json
from dotenv import load_dotenv
from sqlalchemy import create_engine

### Configuración del entorno

In [ ]:

# Cargar las variables de entorno del archivo .env
load_dotenv("/home/jovyan/work/.env")

# SQLAlchemy
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")    
conexion_url = f"mysql+mysqlconnector://{user}:{password}@{host}:{port}/{database}"


engine = create_engine(conexion_url)

# Modelo
llm_url = os.getenv("OLLAMA_URL")
llm_model = os.getenv("LLM_MODEL")

# Precargar modelo en memoria
print("Cargando modelo...")
payload = {
    "model": llm_model,
    "prompt": "",
    "keep_alive": "-1"
}
try: 
    response = requests.post(llm_url, json=payload)
    print("Modelo cargado!")
except Exception as e:
    print(f"Error: no se pudo conectar con Ollama: {e}")

### Funciones principales del agente

In [ ]:
def cargar_system_prompt():
    ruta_prompt = '/home/jovyan/work/notebooks/prompt_sistema.md'
    try: 
        with open(ruta_prompt, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print("Error: no se encontró el archivo de prompt de sistema.")
        return ""

def text_to_sql(pregunta_usuario):    
    system_prompt = cargar_system_prompt()
    if not system_prompt:
        raise Exception("No se pudo cargar el prompt del sistema.")

    full_prompt = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
        f"Traduce esta pregunta a SQL: {pregunta_usuario}<|eot_id|><|start_header_id|>" 
        f"assistant<|end_header_id|>\n\n"
    )
    
    payload = {
        "model": llm_model,
        "prompt": full_prompt,
        "stream": True,
        "options": {
            "temperature": 0.0 # Precisión estricta
        }
    }
    
    response = requests.post(llm_url, json=payload, stream=True)
    sql_acumulado = ""

    for line in response.iter_lines():
        if line:
            chunk = json.loads(line.decode('utf-8'))
            texto_pedazo = chunk.get('response', '')
            print(texto_pedazo, end="", flush=True)
            sql_acumulado += texto_pedazo
    return sql_acumulado.strip()

   
def ejecutar_consulta(sql):
    try:
        with engine.connect() as conn:
            df = pd.read_sql(sql, conn)
        return df
    except Exception as e:
        raise e


def preguntar_al_agente(pregunta):
    print(f"Pregunta del usuario: \n> {pregunta}\n")
    try:
        print(f"SQL Generado por el modelo:",end="\n", flush=True)
        sql = text_to_sql(pregunta)
        print("\n")
        
        print("Conectando a la base de datos...")
        df_resultado = ejecutar_consulta(sql)

        print("Resultado:")
        display(df_resultado)
        
    except Exception as e:
        print(f"Ocurrió un error:\n{e}\n\n")

## Chatear

In [ ]:
print("¡Hola! ¡Soy el Agente BiblioIA! Escribí 'salir' para terminar.")
print("-" * 60)

while True:
    pregunta = input("\nIngresá tu pregunta: \n > ")
    if pregunta.lower() in ['salir', 'chau','adios','adiós','exit','quit']:
        print("¡Nos vemos!")
        break
    if pregunta.strip() == "":
        continue
        
    preguntar_al_agente(pregunta)

## Sección de pruebas

In [ ]:
print("Prueba rápida de IA")
print("-" * 60)

# preguntar_al_agente("cuáles son los 10 libros más prestados de medicina?")
# preguntar_al_agente("Hola!")
preguntar_al_agente("Qué socios tienen más de 10 préstamos históricos?")

In [ ]:
try:
    res = requests.post(llm_url, 
                        json={"model": "llama3.2", "prompt": "Hola, estás vivo?", "stream": False}, 
                        timeout=60)
    print("Respuesta de Ollama:", res.json()['response'])
except Exception as e:
    print("Error al conectar con Ollama:", e)